# Phase 1: Image Ingestion and Document Flattening
### Simple, Step-by-Step Interactive Guide & Visual Walkthrough
---
### What is Phase 1 and why do we need it?
When someone takes a picture of handwritten notes using a smartphone or camera:
1. **The angle is tilted:** The camera is rarely held completely flat above the page.
2. **Background clutter exists:** The photo often captures the wooden desk, shadows, or items around the paper.

**Phase 1 has three simple tasks:**
- **Task 1 (Safe Loading):** Load the image safely without crashing if a file is missing or corrupted.
- **Task 2 (Straightening/Flattening):** Find the 4 corners of the paper sheet and straighten it out (like looking at it directly from above).
- **Task 3 (Untouched Backup):** Keep an exact original copy of the photo so we can draw our final recognized letters on top in Phase 7.

In [ ]:
# Step 1: Import required standard and computer vision libraries
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model folder to Python path so we can import our Phase 1 functions
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import (
    load_image,
    detect_document_corners,
    warp_perspective,
    ingest_image,
    order_quad_points,
)
import config

print("All Phase 1 libraries and modules loaded successfully!")

---
## Step 2: Loading an Image Safely
In standard beginner code, people use `cv2.imread(filepath)`. But if the file is missing or corrupted, OpenCV silently returns `None` and the program crashes on the next line.

Our `load_image()` function:
1. Checks if the file exists on disk.
2. Verifies that OpenCV loaded real pixel data.
3. Creates an **independent copy** of the original image so it stays untouched for Phase 7.

In [ ]:
# Select a test image from the data/samples directory
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Load the image using our Phase 1 function
original_image, raw_frame = load_image(sample_file)

h, w, c = original_image.shape
print(f"File Loaded : {sample_file.name}")
print(f"Image Width : {w} pixels")
print(f"Image Height: {h} pixels")
print(f"Channels    : {c} (BGR Color)")

---
## Step 3: Finding the 4 Corners of the Paper
How does the computer find a piece of paper on a desk?
1. **Grayscale:** Convert the color photo to black-and-white (`cv2.cvtColor`).
2. **Blur:** Smooth out paper texture and camera sensor noise (`cv2.GaussianBlur`).
3. **Edge Detection:** Find the borders of objects in the scene (`cv2.Canny`).
4. **Find Contours:** Find closed shapes and identify the largest 4-corner shape (`cv2.approxPolyDP`).

In [ ]:
# Automatically detect the 4 corners of the paper
corners = detect_document_corners(raw_frame, min_area_ratio=0.15)

if corners is not None:
    print("Found 4 document corners:")
    names = ["Top-Left", "Top-Right", "Bottom-Right", "Bottom-Left"]
    for name, (x, y) in zip(names, corners):
        print(f"  * {name:<12}: x = {x:6.1f}, y = {y:6.1f}")
else:
    print("No 4-corner paper border detected. Direct pass-through will be used.")

---
## Step 4: Straightening the Paper (Perspective Warp)
Once we know the 4 corners, we use **Homography (`cv2.warpPerspective`)**.

*Analogy:* Imagine holding a tilted sheet of paper in your hands, and physically rotating it so it faces your eyes straight on. That is exactly what this step does mathematically!

In [ ]:
if corners is not None:
    # Straighten and crop the document
    flattened_image, M = warp_perspective(raw_frame, corners)
    print(f"Straightened Document Dimensions: {flattened_image.shape[1]}x{flattened_image.shape[0]} px")
else:
    flattened_image = raw_frame
    print("Bypassed warping (full image used).")

---
## Step 5: Visualizing Before vs. After
Let us display the original camera photo side-by-side with the straightened page.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Original photo with detected green corner lines
display_original = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
if corners is not None:
    pts = corners.astype(np.int32)
    for i in range(4):
        p1 = tuple(pts[i])
        p2 = tuple(pts[(i + 1) % 4])
        cv2.line(display_original, p1, p2, (0, 255, 0), 4)
        cv2.circle(display_original, p1, 10, (255, 0, 0), -1)

axes[0].imshow(display_original)
axes[0].set_title("1. Original Photo (Detected Paper Outline)", fontsize=12, fontweight="bold")
axes[0].axis("off")

# Right: Flattened & straightened paper surface
display_flat = cv2.cvtColor(flattened_image, cv2.COLOR_BGR2RGB)
axes[1].imshow(display_flat)
axes[1].set_title(f"2. Straightened Paper ({flattened_image.shape[1]}x{flattened_image.shape[0]} px)", fontsize=12, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

---
## Step 6: What is "Graceful Bypass"?
What happens if someone takes a tight photo of handwritten notes where the edges of the paper are NOT in the frame?

- Does our code crash? **No!**
- The function detects that there are no paper borders and **automatically bypasses** warping.
- It passes the full image straight to Phase 2 (Preprocessing).

In [ ]:
# Testing with sample_camera_capture.png (close-up photo without paper boundaries)
closeup_sample = Path("../data/samples/sample_camera_capture.png")

orig_close, ing_close, meta_close = ingest_image(closeup_sample)

print(f"Image Name    : {closeup_sample.name}")
print(f"Original Size : {orig_close.shape[1]}x{orig_close.shape[0]} px")
print(f"Ingested Size : {ing_close.shape[1]}x{ing_close.shape[0]} px")
print(f"Warp Applied  : {meta_close['warp_applied']} (Safely bypassed without crashing!)")

---
## Quick Professor & Viva Cheat Sheet (Simple Answers)

> **Q1: What is the main goal of Phase 1?**  
> *Answer:* To load the input camera photo safely, isolate the paper from the desk background, and remove any perspective tilt so the handwriting is straight.
>
> **Q2: Why do we use `cv2.imread` instead of Deep Learning libraries like PyTorch?**  
> *Answer:* Our project specification strictly requires classical computer vision without deep learning frameworks. OpenCV is the industry standard for fast, classical image processing.
>
> **Q3: What happens if the photo has no visible desk or paper corners?**  
> *Answer:* The code includes an automatic *graceful bypass*. If 4 clear corners are not detected, it does not fail; it passes the full photo directly to Phase 2.
>
> **Q4: Why do we keep a copy of the original image?**  
> *Answer:* In Phase 7, we want to draw our final recognized letters and green bounding boxes directly on top of the student's original photo for clear visual verification.